In [66]:
import xarray as xr
import climpred as cp
from xarray import DataArray

In [67]:
range_id = 1
dataset_name = "ecmwf_51"
# Mask and scale is used for scaling factor
dataset = xr.open_dataset(f"{dataset_name}_{range_id}.nc", mask_and_scale=False)
dataset = xr.decode_cf(dataset)

In [68]:
def convert_units(da: DataArray, new_unit: str) -> DataArray:
        da = da.metpy.quantify()
        da = da.metpy.convert_units(new_unit)
        da = da.metpy.dequantify()
        return da

dataset["tprate"] = convert_units(dataset["tprate"], "mm/day")

In [69]:
# Make dataset ready for climpred
renamed_vars = ["indexing_time", "forecast_reference_time"]
for rv in renamed_vars:
    if rv in dataset.coords:
        dataset = dataset.rename({rv : "init"})
dataset = dataset.rename({"forecastMonth" : "lead", "number": "member", "tprate" : "pr"})
dataset["lead"].attrs["units"] = "months"
dataset = dataset.assign_coords(lead=dataset["lead"] - 1)

In [70]:
# Convert from init + lead config to valid_times for leads
model = cp.HindcastEnsemble(dataset).get_initialized()
dataset = cp.utils.convert_init_lead_to_valid_time_lead(model)
dataset

<xarray.Dataset> Size: 3MB
Dimensions:     (lead: 6, member: 25, valid_time: 293, latitude: 4, longitude: 4)
Coordinates:
  * lead        (lead) int64 48B 0 1 2 3 4 5
  * member      (member) int64 200B 0 1 2 3 4 5 6 7 ... 17 18 19 20 21 22 23 24
  * valid_time  (valid_time) object 2kB 1993-01-01 00:00:00 ... 2017-05-01 00...
    init        (lead, valid_time) object 14kB 1993-01-01 00:00:00 ... 2016-1...
  * latitude    (latitude) float64 32B -26.0 -27.0 -28.0 -29.0
  * longitude   (longitude) float64 32B 150.0 151.0 152.0 153.0
Data variables:
    pr          (lead, member, valid_time, latitude, longitude) float32 3MB 4...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts

In [71]:
dataset = dataset.rename({"valid_time" : "time"})
dataset = dataset.drop_vars(["init"])

In [72]:
# Prediction on ensemble median
# NOTE: Having lower prediction value
dataset = dataset.median(dim='member')

In [73]:
dataset.to_netcdf(f"{dataset_name}_{range_id}_processed.nc")

In [65]:
dataset

<xarray.Dataset> Size: 115kB
Dimensions:    (lead: 6, time: 293, latitude: 4, longitude: 4)
Coordinates:
  * lead       (lead) int64 48B 0 1 2 3 4 5
  * time       (time) object 2kB 1993-01-01 00:00:00 ... 2017-05-01 00:00:00
  * latitude   (latitude) float64 32B -26.0 -27.0 -28.0 -29.0
  * longitude  (longitude) float64 32B 150.0 151.0 152.0 153.0
Data variables:
    pr         (lead, time, latitude, longitude) float32 113kB 2.575 ... 1.974
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts